# EMI preclosure: visual data lab

Explore the project's raw loan data and preclosure analysis output with interactive charts. These are dataset-level observations, not a recommendation for an individual borrower.

**Data sources:** `dataset/emi_preclosure_raw_data.csv` and `dataset/emi_preclosure_analysis_results.csv`.

In [ ]:
from importlib.util import find_spec
from pathlib import Path
import sys

required = ("pandas", "plotly")
missing = [package for package in required if find_spec(package) is None]
if missing:
    print(f"Install project dependencies with: {sys.executable} -m pip install -r requirements.txt")
    raise ImportError(f"Missing notebook packages: {', '.join(missing)}")

import pandas as pd
import plotly.express as px

candidates = (Path.cwd(), Path.cwd().parent)
PROJECT_ROOT = next((path for path in candidates if (path / "dataset").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find a dataset/ folder. Open this notebook from the project workspace.")

RAW_PATH = PROJECT_ROOT / "dataset" / "emi_preclosure_raw_data.csv"
RESULTS_PATH = PROJECT_ROOT / "dataset" / "emi_preclosure_analysis_results.csv"
raw = pd.read_csv(RAW_PATH)
results = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else None
print(f"Raw data: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"Analysis results: {results.shape[0]:,} rows x {results.shape[1]} columns" if results is not None else "Analysis results CSV not found; prediction charts will be skipped.")

## Data quality and coverage

In [ ]:
display(raw.head())
print("Missing values by column:")
display(raw.isna().sum().rename("missing").to_frame().query("missing > 0"))
print("Loan types:")
display(raw["Loan_Type"].value_counts(dropna=False).rename("loan_count").to_frame())
if results is not None:
    display(results.head())

## Portfolio mix

In [ ]:
loan_summary = (
    raw.groupby("Loan_Type", dropna=False)
    .agg(
        Loans=("Loan_ID", "count"),
        Total_Outstanding=("Outstanding_Amount", "sum"),
        Average_EMI=("Monthly_EMI", "mean"),
        Average_Rate=("Annual_Interest_Rate", "mean"),
    )
    .sort_values("Total_Outstanding", ascending=False)
)
display(loan_summary.style.format({"Total_Outstanding": "₹{:,.0f}", "Average_EMI": "₹{:,.0f}", "Average_Rate": "{:.2f}%"}))

fig = px.pie(
    loan_summary.reset_index(),
    names="Loan_Type",
    values="Total_Outstanding",
    hole=0.56,
    title="Share of total outstanding by loan type",
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(textposition="inside", texttemplate="%{label}<br>%{percent}", hovertemplate="%{label}<br>₹%{value:,.0f}<extra></extra>")
fig.update_layout(legend_title_text="Loan type", margin=dict(t=65, b=15, l=15, r=15))
fig.show()

## Balances and monthly commitments

In [ ]:
fig = px.box(
    raw, x="Loan_Type", y="Outstanding_Amount", color="Loan_Type",
    points="all", title="Outstanding balance distribution by loan type",
    labels={"Outstanding_Amount": "Outstanding balance (₹)", "Loan_Type": "Loan type"},
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_layout(showlegend=False, yaxis_tickprefix="₹", yaxis_tickformat=",.0f", xaxis_tickangle=-18)
fig.show()

fig = px.scatter(
    raw, x="Monthly_Income", y="Monthly_EMI", color="Loan_Type",
    size="Outstanding_Amount", hover_name="Loan_ID",
    hover_data=["Annual_Interest_Rate", "Remaining_Tenure_Months", "Debt_to_Income_Ratio"],
    title="Monthly EMI in context of borrower income",
    labels={"Monthly_Income": "Monthly income (₹)", "Monthly_EMI": "Monthly EMI (₹)"},
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_layout(xaxis_tickprefix="₹", xaxis_tickformat=",.0f", yaxis_tickprefix="₹", yaxis_tickformat=",.0f")
fig.show()

## Interest rate and estimated preclosure benefit

In [ ]:
benefit_data = results if results is not None else raw
if "Preclosure_Benefit" not in benefit_data.columns:
    raise KeyError("Preclosure_Benefit is missing from both project data files.")

fig = px.scatter(
    benefit_data, x="Annual_Interest_Rate", y="Preclosure_Benefit",
    color="Loan_Type", hover_name="Loan_ID",
    hover_data=[column for column in ("Outstanding_Amount", "Monthly_EMI", "Remaining_Tenure_Months") if column in benefit_data.columns],
    title="Annual interest rate vs. preclosure benefit",
    labels={"Annual_Interest_Rate": "Annual interest rate (%)", "Preclosure_Benefit": "Preclosure benefit (₹)"},
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.add_hline(y=0, line_dash="dash", line_color="#68766f")
fig.update_layout(yaxis_tickprefix="₹", yaxis_tickformat=",.0f")
fig.show()

## Model output: actual vs. predicted

In [ ]:
actual_column = "Preclosure_Benefit"
predicted_column = "Predicted_Preclosure_Benefit"
if results is not None and {actual_column, predicted_column}.issubset(results.columns):
    fig = px.scatter(
        results, x=actual_column, y=predicted_column, color="Loan_Type",
        hover_name="Loan_ID", title="Actual vs. predicted preclosure benefit",
        labels={actual_column: "Actual benefit (₹)", predicted_column: "Predicted benefit (₹)"},
        color_discrete_sequence=px.colors.qualitative.Bold,
    )
    axis_min = min(results[actual_column].min(), results[predicted_column].min())
    axis_max = max(results[actual_column].max(), results[predicted_column].max())
    fig.add_shape(type="line", x0=axis_min, y0=axis_min, x1=axis_max, y1=axis_max, line=dict(color="#52675c", dash="dash"))
    fig.update_layout(xaxis_tickprefix="₹", yaxis_tickprefix="₹", xaxis_tickformat=",.0f", yaxis_tickformat=",.0f")
    fig.show()
    results["Prediction_Error"] = results[actual_column] - results[predicted_column]
    fig = px.histogram(results, x="Prediction_Error", color="Loan_Type", marginal="box", title="Prediction error distribution", labels={"Prediction_Error": "Actual minus predicted (₹)"}, color_discrete_sequence=px.colors.qualitative.Bold)
    fig.add_vline(x=0, line_dash="dash", line_color="#52675c")
    fig.show()
    print(f"Mean absolute error: ₹{results['Prediction_Error'].abs().mean():,.2f}")
    print(f"Median absolute error: ₹{results['Prediction_Error'].abs().median():,.2f}")
else:
    print("Skipped: the analysis results file needs both actual and predicted benefit columns.")

## Quick dataset readout

In [ ]:
print(f"Loans in raw dataset: {len(raw):,}")
print(f"Loan types represented: {raw['Loan_Type'].nunique()}")
print(f"Median outstanding balance: ₹{raw['Outstanding_Amount'].median():,.0f}")
print(f"Median annual interest rate: {raw['Annual_Interest_Rate'].median():.2f}%")
print(f"Rows with positive preclosure benefit: {(raw['Preclosure_Benefit'] > 0).mean():.1%}")
print("")
print("Median estimated benefit by loan type:")
display(raw.groupby("Loan_Type")["Preclosure_Benefit"].median().sort_values(ascending=False).rename("median_benefit").to_frame().style.format("₹{:,.2f}"))

if results is not None and predicted_column in results.columns:
    print(f"Median absolute prediction error: ₹{(results[actual_column] - results[predicted_column]).abs().median():,.2f}")